## Лабораторная работа 15.
Задание:
Сделать визуализацию работы Resnet50 (как на видео https://www.youtube.com/watch?v=fZvOy0VXWAI&t=85s)
1. Установить torch, torchvision, скачать Resnet (pretrained=True)
2. Написать функцию get_features_map() (установить hook), которая "возвращает" результаты работы слоя layer4 (находится перед слоем avgpool)
3. Достать из модели матрицу весов "W" последнего (полносвязного) слоя (fc).
4. Сложить карты признаков (2048 штук для ResNet50), с коэффициентами w_i, как показано в "статье" - https://alexisbcook.github.io/posts/global-average-pooling-layers-for-object-localization/

In [ ]:
python -c 'from keras.applications.resnet50 import ResNet50; ResNet50().summary()'

SyntaxError: invalid syntax (2337233249.py, line 1)

In [ ]:
# simple implementation of CAM in PyTorch for the networks such as ResNet, DenseNet, SqueezeNet, Inception
# last update by BZ, June 30, 2021 - ADAPTED FOR NEWER PYTORCH

import io
from PIL import Image
from torchvision import models, transforms
import torch
import numpy as np
import cv2
import json
import os

LABELS_file = "/home/julia/Desktop/code/PAC/dogs/n02088466_10083"
image_file = "/home/julia/Desktop/code/PAC/dogs/n02088466_10083.jpg"

if not os.path.exists(LABELS_file):
    print(f"Файл с метками не найден: {LABELS_file}")
    print("Загружаем стандартные метки ImageNet...")

    import urllib.request
    # короче на случай если файл с метками который я скеачала окажется побитым то пможно
    # скачать стандартные метки из imagenet, там должны быть сотоветсивя между индексами класса и названиями объектов

    url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
    urllib.request.urlretrieve(url, "imagenet_classes.json")
    LABELS_file = "imagenet_classes.json"

# networks such as googlenet, resnet, densenet already use global average pooling at the end, so CAM could be used directly.
model_id = 2
if model_id == 1:
    net = models.squeezenet1_1(pretrained=True)
    finalconv_name = "features"  # this is the last conv layer of the network
elif model_id == 2:
    net = models.resnet50(pretrained=True)
    finalconv_name = "layer4"
elif model_id == 3:
    net = models.densenet161(pretrained=True)
    finalconv_name = "features"

net.eval()
# это режим оценки модели, он отключает dropout, batch normalization
# то есть модель не дообучается на этой картинке а использует то что она уже знает чтобы понять что перед ней


# hook the feature extractor
features_blobs = []


def hook_feature(module, input, output):
    features_blobs.append(output.detach().cpu().numpy())
    # output - то что вышло из последнего слоя те 2048 картинок 7*7
    # ddetach - отключить граф вычислений  то есть не запоминать последовательность действий
    # чтобы использовать меньше памяти и предотвратить случсайнфе изменения
    # переложили на процессор потому что нампай не читает с видеокарты


# тут проходимся по всем слоям сети находим нужный и к нему приставляем хук
# теперь каждый раз когда данные пройдут через этот слой хук сработает
# Register hook properly for newer PyTorch
for name, module in net.named_modules():
    if name == finalconv_name:
        module.register_forward_hook(hook_feature)
        break

# get the softmax weight
params = list(net.parameters())  # net.parameters() - все обучаемые параметры сети
weight_softmax = np.squeeze(params[-2].detach().cpu().numpy())
# берем предпоследний параметр. последний те -1 это смещение
# а -2 это должны быть веса
# убираем с них лишние размерности
# вообще  weight_param: torch.Size([1000, 2048]) и лишний тут как бы и нет но код писали и для других моеделей вдруг там есть
# убирает все размерности где размер = 1

def returnCAM(feature_conv, weight_softmax, class_idx):
    # generate the class activation maps upsample to 256x256
    size_upsample = (256, 256) # увеличить размер итоговой карты
    bz, nc, h, w = feature_conv.shape 
    #  bz - batch size обычно 1
    # # nc - number of channels (2048)
    # h, w - height и width (7 и 7)
    output_cam = []
    #для каждого запрошенного класса своя карта
    for idx in class_idx:
        cam = weight_softmax[idx].dot(feature_conv.reshape((nc, h * w)))
        # перемножение картинок и весов и сложение этого всенго
        cam = cam.reshape(h, w)
        cam = cam - np.min(cam)
        cam_img = cam / np.max(cam)
        cam_img = np.uint8(255 * cam_img)
        # нормализации для визуализации
        output_cam.append(cv2.resize(cam_img, size_upsample))
    return output_cam


normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
preprocess = transforms.Compose(
    [transforms.Resize((224, 224)), transforms.ToTensor(), normalize]
)

# load test image
img_pil = Image.open(image_file).convert("RGB")
img_tensor = preprocess(img_pil)
img_batch = img_tensor.unsqueeze(0) # было [3, 224, 224], стало [1, 3, 224, 224]

with torch.no_grad():
    # опять режим только для ответа. прогоняем изображение 
    logit = net(img_batch)

# load the imagenet category list
with open(LABELS_file, "r") as f:
    if LABELS_file.endswith(".json"):
        classes_data = json.load(f)
        # Обрабатываем разные форматы JSON
        if isinstance(classes_data, dict):
            if all(k.isdigit() for k in classes_data.keys()):
                # Формат {"0": ["n01440764", "tench"], ...}
                classes = [item[1] for item in classes_data.values()]
            else:
                # Формат {"tench": "n01440764", ...}
                classes = list(classes_data.keys())
        elif isinstance(classes_data, list):
            classes = classes_data
    else:
        # Если это текстовый файл с одной меткой на строку
        classes = [line.strip() for line in f.readlines()]

#  получение предсказаний
h_x = torch.nn.functional.softmax(logit, dim=1)[0]
probs, idx = torch.sort(h_x, descending=True)
probs = probs.numpy()
idx = idx.numpy()

# output the prediction
print("\nTop 5 predictions:")
for i in range(0, 5):
    if idx[i] < len(classes):
        print("{:.3f} -> {}".format(probs[i], classes[idx[i]]))
    else:
        print("{:.3f} -> Class #{}".format(probs[i], idx[i]))



# CAM
# generate class activation mapping for the top1 prediction
if features_blobs:
    CAMs = returnCAM(features_blobs[0], weight_softmax, [idx[0]])

    # render the CAM and output
    if idx[0] < len(classes):
        print("\nOutput CAM.jpg for the top1 prediction: %s" % classes[idx[0]])
    else:
        print("\nOutput CAM.jpg for the top1 prediction: Class #%d" % idx[0])

    # Load the original image
    img = cv2.imread(image_file)
    if img is not None:
        height, width, _ = img.shape
        heatmap = cv2.applyColorMap(
            cv2.resize(CAMs[0], (width, height)), cv2.COLORMAP_JET
        )
        result = heatmap * 0.3 + img * 0.5
        cv2.imwrite("CAM.jpg", result)
        print("CAM.jpg saved successfully!")
    else:
        print(f"Could not load image: {image_file}")
else:
    print("No features captured!")

/home/julia/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/julia/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Всего параметров: 161
Тип params[-2]: <class 'torch.nn.parameter.Parameter'>
Размерность weight_param: torch.Size([1000, 2048])
Размерность bias_param: torch.Size([1000])

Top 5 predictions:
0.976 -> Class #163
0.012 -> Class #165
0.009 -> Class #161
0.001 -> Class #168
0.001 -> Class #164

Output CAM.jpg for the top1 prediction: Class #163
CAM.jpg saved successfully!


In [ ]:
"""
Real-time CAM (Class Activation Mapping) from webcam
Based on the CAM implementation for single images
"""

import cv2
import torch
import numpy as np
from torchvision import models, transforms
import json
import urllib.request
import os
from PIL import Image
import time

# ==================== ИНИЦИАЛИЗАЦИЯ МОДЕЛИ (как в первом коде) ====================

print("Инициализация модели ResNet-50...")
# Загружаем метки ImageNet
labels_url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
try:
    with urllib.request.urlopen(labels_url) as response:
        imagenet_data = json.loads(response.read().decode())
        classes = [item[1] for item in imagenet_data.values()]
    print(f"Загружено {len(classes)} классов")
except Exception as e:
    print(f"Ошибка загрузки меток: {e}")
    # Запасной вариант - просто индексы
    classes = [f"Class #{i}" for i in range(1000)]

# Загружаем предобученную ResNet-50
net = models.resnet50(pretrained=True)
finalconv_name = "layer4"
net.eval()

# Хук для захвата признаков
features_blobs = []

def hook_feature(module, input, output):
    features_blobs.append(output.detach().cpu().numpy())

# Регистрируем хук
for name, module in net.named_modules():
    if name == finalconv_name:
        module.register_forward_hook(hook_feature)
        print(f"Хук зарегистрирован на слое: {name}")
        break

# Получаем веса последнего слоя
params = list(net.parameters())
weight_softmax = np.squeeze(params[-2].detach().cpu().numpy())
print(f"Размерность весов: {weight_softmax.shape}")

# Функция создания CAM
def returnCAM(feature_conv, weight_softmax, class_idx):
    size_upsample = (256, 256)
    bz, nc, h, w = feature_conv.shape
    output_cam = []
    for idx in class_idx:
        cam = weight_softmax[idx].dot(feature_conv.reshape((nc, h * w)))
        cam = cam.reshape(h, w)
        cam = cam - np.min(cam)
        cam_img = cam / np.max(cam)
        cam_img = np.uint8(255 * cam_img)
        output_cam.append(cv2.resize(cam_img, size_upsample))
    return output_cam

# Предобработка изображения
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

# ==================== ФУНКЦИЯ ОБРАБОТКИ ОДНОГО КАДРА ====================

def process_frame(frame):
    """
    Обрабатывает один кадр с веб-камеры и возвращает:
    - frame_with_cam: кадр с наложенной CAM
    - top_predictions: список топ-3 предсказаний (класс, вероятность)
    """
    global features_blobs
    
    # Очищаем предыдущие захваченные признаки
    features_blobs = []
    
    # Конвертируем OpenCV BGR в RGB и в PIL Image
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(frame_rgb)
    
    # Предобработка для сети
    img_tensor = preprocess(img_pil)
    img_batch = img_tensor.unsqueeze(0)
    
    # Прогоняем через сеть
    with torch.no_grad():
        logit = net(img_batch)
    
    # Получаем предсказания
    h_x = torch.nn.functional.softmax(logit, dim=1)[0]
    probs, idx = torch.sort(h_x, descending=True)
    probs = probs.numpy()[:3]  # Берем топ-3
    idx = idx.numpy()[:3]
    
    # Формируем список предсказаний
    predictions = []
    for i in range(3):
        class_name = classes[idx[i]] if idx[i] < len(classes) else f"Class #{idx[i]}"
        predictions.append((class_name, probs[i]))
    
    # Создаем CAM для лучшего предсказания
    frame_with_cam = frame.copy()
    if features_blobs and len(features_blobs) > 0:
        CAMs = returnCAM(features_blobs[0], weight_softmax, [idx[0]])
        
        # Нормализуем CAM до размера кадра
        height, width = frame.shape[:2]
        heatmap = cv2.applyColorMap(
            cv2.resize(CAMs[0], (width, height)), 
            cv2.COLORMAP_JET
        )
        
        # Накладываем CAM на оригинальный кадр
        frame_with_cam = cv2.addWeighted(heatmap, 0.3, frame, 0.7, 0)
    
    return frame_with_cam, predictions

# ==================== ОСНОВНОЙ ЦИКЛ ЗАХВАТА ВИДЕО ====================

def main():
    # Открываем веб-камеру (0 - обычно встроенная камера, 1 - внешняя)
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Ошибка: Не удалось открыть камеру")
        return
    
    print("Нажмите 'q' для выхода")
    print("Нажмите 's' для сохранения текущего кадра с CAM")
    
    # Для подсчета FPS
    fps_counter = 0
    fps_time = time.time()
    fps_display = 0
    
    frame_count = 0
    
    while True:
        # Захватываем кадр
        ret, frame = cap.read()
        if not ret:
            print("Ошибка захвата кадра")
            break
        
        # Обрабатываем каждый 2-й кадр для экономии ресурсов (можно настроить)
        # Это уменьшит нагрузку на процессор, но CAM будет обновляться реже
        process_this_frame = (frame_count % 2 == 0)
        
        if process_this_frame:
            # Обрабатываем кадр
            frame_with_cam, predictions = process_frame(frame)
            
            # Обновляем FPS
            fps_counter += 1
            if time.time() - fps_time >= 1.0:
                fps_display = fps_counter
                fps_counter = 0
                fps_time = time.time()
        else:
            # Используем предыдущий обработанный кадр
            # (нужно сохранять frame_with_cam и predictions от предыдущего раза)
            if 'frame_with_cam' not in locals():
                frame_with_cam, predictions = process_frame(frame)
        
        # Добавляем информацию на кадр
        display_frame = frame_with_cam.copy()
        
        # Рисуем полупрозрачный фон для текста
        overlay = display_frame.copy()
        cv2.rectangle(overlay, (5, 5), (400, 140), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.5, display_frame, 0.5, 0, display_frame)
        
        # Выводим топ-3 предсказания
        y_offset = 30
        cv2.putText(display_frame, "Top predictions:", (10, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        for i, (class_name, prob) in enumerate(predictions):
            y_offset += 25
            text = f"{i+1}. {class_name}: {prob:.2%}"
            cv2.putText(display_frame, text, (10, y_offset), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Выводим FPS
        cv2.putText(display_frame, f"FPS: {fps_display}", (10, y_offset + 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
        # Выводим инструкции
        cv2.putText(display_frame, "Q: quit | S: save", (10, y_offset + 50), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 255, 100), 1)
        
        # Показываем результат
        cv2.imshow('Real-time CAM', display_frame)
        
        # Обработка клавиш
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('s'):
            # Сохраняем текущий кадр
            timestamp = int(time.time())
            filename = f"cam_capture_{timestamp}.jpg"
            cv2.imwrite(filename, display_frame)
            print(f"Кадр сохранён как {filename}")
        
        frame_count += 1
    
    # Освобождаем ресурсы
    cap.release()
    cv2.destroyAllWindows()
    print("Программа завершена")

if __name__ == "__main__":
    main()